<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Case study: route banking questions

**[Case study: route banking questions](https://learning.nextia-ai.com/courses/text-routing/m10/route-banking-questions/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 10, case study 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** take 13,083 real-looking banking questions in 77 fine-grained intents from the first look to a routing decision: a written mapping to 7 teams, four method families compared on one measure, a review threshold chosen from the cost table, one test, and a run on new questions.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page. The notebook runs in a few minutes after the install. |
| **Needs** | An internet connection for the setup cells: the data (1 MB), the course's files (44 MB) and the base model (about 470 MB, once). No account and no key: the LLM's answers were recorded. |
| **You should know** | The cost table, from [The cost of each mistake](https://learning.nextia-ai.com/courses/text-routing/m01/the-cost-of-each-mistake/); fine-tuning, from [Fine-tune a small encoder on CPU](https://learning.nextia-ai.com/courses/text-routing/m04/fine-tune-a-small-encoder-on-cpu/); team-then-topic labels, from [Hierarchical labels](https://learning.nextia-ai.com/courses/text-routing/m07/hierarchical-labels/). |
| **Data** | Banking77 by PolyAI, CC BY 4.0. Details in `dataset.md` next to this notebook. |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, sentence-transformers 6.2.0, Transformers 5.22.2 and PyTorch 2.15.0 (CPU), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m10/route-banking-questions/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M10-L01-route-banking-questions/route-banking-questions-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, sentence-transformers 6.2.0, transformers 5.22.2, torch 2.15.0, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2 sentence-transformers==6.2.0 transformers==5.22.2 torch==2.15.0
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import sentence_transformers; print("sentence-transformers", sentence_transformers.__version__)
import transformers; print("transformers", transformers.__version__)
import torch; print("torch", torch.__version__)

## Setup: get the data and the course's files

Run the next cell. It downloads into a folder `banking77`:

- Banking77's two files, `train.csv` and `test.csv` (1 MB), from PolyAI's repository at a fixed version;
- the course's files: the written mapping of the 77 intents to 7 routes, the recorded LLM answers, the fine-tuned encoder (43 MB) and the helper `routing.py`.

It checks each file's checksum. You should see `Ready: 8 files in banking77/`. If you run the cell again, it uses the files that are already there.

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `banking77`, then run the cell again.

**Source and licence.** "Banking77" by Iñigo Casanueva, Tadas Temčinas, Daniela Gerz, Matthew Henderson and Ivan Vulić (PolyAI), [github.com/PolyAI-LDN/task-specific-datasets](https://github.com/PolyAI-LDN/task-specific-datasets), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The questions were written for an imagined bank: no real customers. `dataset.md` lists what we changed.

In [ ]:
# Setup: download the data and the case study's files into banking77/ and check each file.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

FILES = {
    "data/train.csv": [
        "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/train.csv",
        "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b"
    ],
    "data/test.csv": [
        "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/test.csv",
        "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d"
    ],
    "data/intent_routes.csv": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L01-route-banking-questions/data/intent_routes.csv",
        "378641d359d3d43be5af1fad7f11050214838968e666199838f42bd39db11ac4"
    ],
    "recordings/llm-few-shot.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L01-route-banking-questions/recordings/llm-few-shot.jsonl",
        "071904ff0722bd2a9e77a2c80a4ea31266164c1878741bda56e170c1de182a5e"
    ],
    "recordings/llm-zero-shot.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L01-route-banking-questions/recordings/llm-zero-shot.jsonl",
        "b51df82e1e926ce8e852a0ece028064b29d23e0cd1b7743c1a7c915c7a699023"
    ],
    "models/banking-encoder/labels.json": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L01-route-banking-questions/models/banking-encoder/labels.json",
        "c915a9dc7b186a0760d6e119d0d6be303e8205c9ec9da8d097c8e874bfe06677"
    ],
    "models/banking-encoder/changed_weights.safetensors": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L01-route-banking-questions/models/banking-encoder/changed_weights.safetensors",
        "f4cfdc7a1f46f9a1f54c7813884fcaf92de66c1dd7cf8e7115c071aaddbbbc64"
    ],
    "routing.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/routing.py",
        "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c"
    ],
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("banking77").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder banking77 and run again.")
os.chdir(PROJECT)
print(f"Ready: {len(FILES)} files in {PROJECT.name}/")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, PyTorch and the course helper as `rt`, and defines the check helpers. They tell you if your answer is right, without showing it:

- `expect(what, yours, expected)` compares one of your numbers with the expected one.
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the libraries, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
import json, time, torch
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data and the first checks

One row is one question and its intent. The next cell loads both files, numbers the rows and runs the first checks.

> **Check.** You should see 10,003 training questions, 3,080 test questions and 77 intents, with no missing values and no repeated questions. Two intent names are odd: `Refund_not_showing_up` and `reverted_card_payment?`. We keep them as they are.

In [ ]:
raw_train = pd.read_csv("data/train.csv").rename(columns={"category": "intent"})
test = pd.read_csv("data/test.csv").rename(columns={"category": "intent"})
# Banking77 has no message IDs: we number the rows in the order of the files.
raw_train.insert(0, "message_id", [f"B-TR-{i:05d}" for i in range(1, len(raw_train) + 1)])
test.insert(0, "message_id", [f"B-TE-{i:04d}" for i in range(1, len(test) + 1)])
print(f"train file: {len(raw_train):,} questions | test file: {len(test):,} | intents: {raw_train.intent.nunique()}")
print(f"missing values: {raw_train.isna().sum().sum() + test.isna().sum().sum()} | repeated questions: "
      f"{raw_train.text.duplicated().sum()} | test questions also in train: {test.text.isin(raw_train.text).sum()}")
words = raw_train.text.str.split().str.len()
print(f"words per question: median {words.median():.0f}, 90% have at most {words.quantile(0.9):.0f}, longest {words.max()}")
per_intent = raw_train.intent.value_counts()
print(f"train questions per intent: from {per_intent.min()} ({per_intent.idxmin()}) to {per_intent.max()} "
      f"({per_intent.idxmax()}) | test: {test.intent.value_counts().min()} to {test.intent.value_counts().max()}")
print("odd intent names:", [i for i in sorted(per_intent.index) if not i.replace("_", "").isalpha() or not i.islower()])

The questions are short: half have 10 words or fewer. The training file is not balanced (35 to 187 questions per intent), but the test file has exactly 40 per intent.

## 2. The written mapping: 77 intents, 7 routes

A bank does not have 77 teams. We wrote down which team answers each intent, in `data/intent_routes.csv`. The `security` route is urgent: someone else may be using the customer's money. The lesson page explains each choice.

In [ ]:
mapping = pd.read_csv("data/intent_routes.csv")
ROUTE_OF = dict(zip(mapping.intent, mapping.route))
ROUTES = list(dict.fromkeys(mapping.route))  # the 7 routes, in the order of the file
URGENT = set(mapping.route[mapping.priority == "urgent"])
missing = set(raw_train.intent) - set(ROUTE_OF)
print(f"{len(ROUTE_OF)} intents mapped to {len(ROUTES)} routes; intents without a route: {len(missing)}")
for part in (raw_train, test):
    part["route"] = part.intent.map(ROUTE_OF)
summary = mapping.groupby("route", sort=False).agg(intents=("intent", "size"), priority=("priority", "first"))
summary["train_questions"] = raw_train.route.value_counts()
summary["share"] = (summary.train_questions / len(raw_train)).round(3)
summary

> **Check.** You should see 7 routes and no intent without a route. `cards` has the most intents (20), and `security` the fewest (6).

## 3. The split

We take 2,000 validation questions out of the training file. We choose the methods and the threshold on them. The test file stays closed until section 12.

In [ ]:
from sklearn.model_selection import train_test_split
# 2,000 of the training questions become the validation set: we choose methods and thresholds on it.
# The test file stays closed until the end, and we use it once.
train, validation = train_test_split(raw_train, test_size=2000, stratify=raw_train.intent, random_state=36)
train, validation = train.reset_index(drop=True), validation.reset_index(drop=True)
print(f"train {len(train):,} | validation {len(validation):,} | test {len(test):,} (used once, at the end)")

## 4. The case study's helpers

`score()` returns one row of the comparison table, with the course's columns. It uses the course's cost table from `routing.py` (an **assumption**): a question in the wrong queue costs 4 dollars, an urgent `security` question in the wrong queue 20 dollars, and a review by a person 1.50 dollars.

`route_proba()` turns the 77 intent probabilities into 7 route probabilities: it adds the probabilities of the intents of each route. Run both cells.

In [ ]:
# The case study's helpers: the course's cost table (routing.py) applied to the bank's 7 routes.
from sklearn.metrics import f1_score
rt.ROUTES = ROUTES                       # routing.py scores these routes
CPU_HOUR = 0.05                          # dollars per CPU-hour (the course's assumption)
INTENTS = sorted(mapping.intent)

def truth_of(part):
    """The columns that routing.py needs: route, topic (here the intent), priority and language."""
    return pd.DataFrame({"route": part.route.values, "topic": part.intent.values, "language": "en",
                         "priority": ["urgent" if r in URGENT else "normal" for r in part.route]}, index=part.index)

def score(name, part, routes, intents=None, run_per_1000=0.0, ms=None):
    """One row of the comparison table for a part (validation or test) and the predicted routes.
    A route can be "review": a person reads the question first (1.50 dollars)."""
    row = rt.score(name, truth_of(part), list(routes), run_per_1000=run_per_1000, ms_per_message=ms)
    routes = np.asarray(routes)
    sec = (part.route == "security").to_numpy()
    out = {"method": name, "intent_accuracy": None, "intent_macro_f1": None,
           "route_accuracy": row["accuracy"], "route_macro_f1": row["macro_f1"],
           "security_recall": round(float((routes[sec] == "security").mean()), 3),
           "review_share": row["review_share"], "mistakes_per_1000": row["mistakes_per_1000"],
           "run_per_1000": row["run_per_1000"], "total_per_1000": row["total_per_1000"],
           "ms_per_message": row["ms_per_message"]}
    if intents is not None:
        out["intent_accuracy"] = round(float((np.asarray(intents) == part.intent.values).mean()), 3)
        out["intent_macro_f1"] = round(float(f1_score(part.intent, intents, average="macro", labels=INTENTS)), 3)
    return out

def table(rows):
    """The rows as a table, cheapest total first."""
    return pd.DataFrame(rows).sort_values("total_per_1000").reset_index(drop=True)

def route_proba(proba, classes):
    """From the 77 intent probabilities to 7 route probabilities: add the intents of each route."""
    P = pd.DataFrame(proba, columns=classes)
    return P.T.groupby(P.columns.map(ROUTE_OF)).sum().T[ROUTES]

def cpu_cost(ms):
    return ms / 1000 / 3600 * CPU_HOUR * 1000   # dollars per 1,000 messages at `ms` milliseconds each

def ms_per_message(predict, texts, n=200):
    t0 = time.perf_counter()
    for t in texts[:n]:
        predict([t])
    return (time.perf_counter() - t0) / n * 1000

In [ ]:
# The hosted LLM's answers were recorded once for a sample: up to 10 questions per intent.
validation_sample = validation.sample(frac=1, random_state=36).groupby("intent").head(10).sort_index()
test_sample = test.sample(frac=1, random_state=36).groupby("intent").head(10).sort_index()
print(f"samples with recorded LLM answers: validation {len(validation_sample)}, test {len(test_sample)}")

## 5. The baseline: TF-IDF and logistic regression

The course project's TF-IDF pipeline learns the 77 intents. The route is the route with the highest added probability. The table also has "everything to a person": the cost if a person read every question.

> **Check.** You should see an intent accuracy of about 0.896 and a route accuracy of about 0.962, for 256.0 dollars per 1,000 questions. Everything to a person costs 1,500.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, make_pipeline

def tfidf_lr():
    """The course project's TF-IDF + logistic regression (larkroute.tfidf): words and word pairs, plus letter groups."""
    words = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")
    letters = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)
    return make_pipeline(FeatureUnion([("words", words), ("letters", letters)]), LogisticRegression(C=10, max_iter=3000))

t0 = time.perf_counter()
tfidf = tfidf_lr().fit(train.text, train.intent)          # learns the 77 intents
print(f"fitted in {time.perf_counter() - t0:.0f} s")
tfidf_val = route_proba(tfidf.predict_proba(validation.text), tfidf.classes_)
tfidf_intent_val = tfidf.predict(validation.text)
tfidf_ms = ms_per_message(tfidf.predict_proba, list(validation.text))
rows_val = [score("everything to a person", validation, ["review"] * len(validation)),
            score("TF-IDF + LR", validation, tfidf_val.idxmax(axis=1), tfidf_intent_val,
                  cpu_cost(tfidf_ms), tfidf_ms)]
table(rows_val)

## 6. Your turn: learn the routes directly?

Why learn 77 intents if we only need 7 routes? Train the same pipeline on `train.route` instead of `train.intent`, and measure its route accuracy on the validation questions.

> **Predict.** Is the route accuracy higher or lower than the baseline's?

In [ ]:
direct = ...           # the same pipeline, trained on train.route instead of train.intent
direct_accuracy = ...  # its route accuracy on the validation questions, rounded to 3 decimals

In [ ]:
expect('direct_accuracy', direct_accuracy, 0.944, tolerance=0.002)

## 7. Where the intents get confused

The next cell counts the baseline's mistakes on the validation questions, by pair of intents. `same_route` is `True` when both intents go to the same team: then the mistake costs nothing.

In [ ]:
# Which intents does the baseline mix up? Count (true intent, predicted intent) pairs on the validation questions.
pairs = pd.DataFrame({"intent": validation.intent, "predicted": tfidf_intent_val})
pairs = pairs[pairs.intent != pairs.predicted].value_counts().rename("questions").reset_index()
pairs["same_route"] = pairs.intent.map(ROUTE_OF) == pairs.predicted.map(ROUTE_OF)
wrong = pairs.questions.sum()
print(f"intent mistakes: {wrong}, of which inside the same route: {pairs.questions[pairs.same_route].sum()} "
      f"({pairs.questions[pairs.same_route].sum() / wrong:.0%}); they cost nothing at the route level")
pairs.head(10)

Most intent mistakes stay inside one route. The lesson page discusses the best-known pair of Banking77: a card that has not come (`card_arrival`) and a question about when it will come (`card_delivery_estimate`).

> **Your turn.** Count the validation questions labelled `card_delivery_estimate` that the baseline predicts as `card_arrival`. Use `pair` and `tfidf_intent_val`.

In [ ]:
both = ["card_arrival", "card_delivery_estimate"]
pair = validation[validation.intent.isin(both)]
estimate_as_arrival = ...   # validation questions labelled card_delivery_estimate that the baseline predicts as card_arrival

In [ ]:
expect('estimate_as_arrival', estimate_as_arrival, 2)

## 8. Frozen embeddings and logistic regression

The course's older embedding model turns each question into 384 numbers; a logistic regression learns the intents on them. The cell embeds all 13,083 questions: about a minute on a laptop, plus the model download the first time.

> **Check.** You should see a route accuracy of about 0.962: lower than TF-IDF.

In [ ]:
from sentence_transformers import SentenceTransformer
torch.set_num_threads(4)
# The course's older embedding model, frozen; a logistic regression learns the 77 intents on its 384 numbers.
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                               revision="e8f8c211226b894fcb81acc59f3b34ba3efd5f42", device="cpu")
t0 = time.perf_counter()
E = {name: embedder.encode(list(part.text), batch_size=64, normalize_embeddings=True)
     for name, part in [("train", train), ("validation", validation), ("test", test)]}
print(f"{sum(len(v) for v in E.values()):,} questions embedded in {time.perf_counter() - t0:.0f} s")
frozen = LogisticRegression(C=10, max_iter=3000).fit(E["train"], train.intent)
frozen_val = route_proba(frozen.predict_proba(E["validation"]), frozen.classes_)
frozen_ms = ms_per_message(lambda t: frozen.predict_proba(embedder.encode(t, normalize_embeddings=True)),
                           list(validation.text), n=100)
rows_val.append(score("embeddings (frozen) + LR", validation, frozen_val.idxmax(axis=1),
                      frozen.predict(E["validation"]), cpu_cost(frozen_ms), frozen_ms))
table(rows_val)

## 9. The fine-tuned encoder

The same model, fine-tuned on the 8,003 training questions: its layers learned the 77 intents. The training took about 10 minutes on our laptop, so the next cell loads the ready checkpoint (only the weights that changed). The cell after it can train the model again, if you want.

> **Check.** `missing` lists only the word table, and `unexpected` is empty. The route accuracy is about 0.968 and the cost about 182.0 dollars per 1,000: the best so far.

In [ ]:
import json
from safetensors.torch import load_file
from transformers import AutoModelForSequenceClassification, AutoTokenizer
# The ready checkpoint: the encoder fine-tuned on the 8,003 training questions (finetune.py, 4 epochs).
meta = json.load(open("models/banking-encoder/labels.json"))
tok = AutoTokenizer.from_pretrained(meta["base_model"], revision=meta["revision"])
encoder = AutoModelForSequenceClassification.from_pretrained(meta["base_model"], revision=meta["revision"],
                                                             num_labels=len(meta["intents"]))
weights = {k: v.float() for k, v in load_file("models/banking-encoder/changed_weights.safetensors").items()}
missing, unexpected = encoder.load_state_dict(weights, strict=False)
print("missing:", missing, "| unexpected:", unexpected)
encoder.eval()

@torch.no_grad()
def encoder_proba(texts, size=64):
    out = []
    for i in range(0, len(texts), size):
        enc = tok(list(texts[i:i + size]), truncation=True, max_length=meta["max_length"], padding=True, return_tensors="pt")
        out.append(torch.softmax(encoder(**enc).logits, -1).numpy())
    return np.concatenate(out)

enc_p_val = encoder_proba(list(validation.text))
enc_val = route_proba(enc_p_val, meta["intents"])
enc_ms = ms_per_message(encoder_proba, list(validation.text), n=100)
rows_val.append(score("fine-tuned encoder", validation, enc_val.idxmax(axis=1),
                      np.array(meta["intents"])[enc_p_val.argmax(axis=1)], cpu_cost(enc_ms), enc_ms))
table(rows_val)

In [ ]:
TRAIN_FULL = False  # change to True to fine-tune the encoder yourself: about 10 minutes on our laptop, longer on Colab
if TRAIN_FULL:
    import math
    lab = {x: i for i, x in enumerate(meta["intents"])}
    tok = AutoTokenizer.from_pretrained(meta["base_model"], revision=meta["revision"])
    mine = AutoModelForSequenceClassification.from_pretrained(meta["base_model"], revision=meta["revision"],
                                                              num_labels=len(lab))
    mine.base_model.embeddings.word_embeddings.weight.requires_grad = False  # freeze the word table
    texts, y = list(train.text), [lab[x] for x in train.intent]
    opt = torch.optim.AdamW([p for p in mine.parameters() if p.requires_grad], lr=0.0002, weight_decay=0.01)
    steps = math.ceil(len(texts) / 32) * 6
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 50) * max(0.0, 1 - s / steps))
    torch.manual_seed(36)
    for epoch in range(6):
        mine.train()
        order = np.random.default_rng(36 + epoch).permutation(len(texts))
        for i in range(0, len(order), 32):
            j = order[i:i + 32]
            enc = tok([texts[k] for k in j], truncation=True, max_length=meta["max_length"], padding=True, return_tensors="pt")
            mine(**enc, labels=torch.tensor([y[k] for k in j])).loss.backward()
            torch.nn.utils.clip_grad_norm_(mine.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        print(f"epoch {epoch + 1} done")
    mine.eval()
    with torch.no_grad():
        enc = tok(list(validation.text), truncation=True, max_length=meta["max_length"], padding=True, return_tensors="pt")
        mine_p = torch.softmax(mine(**enc).logits, -1).numpy()
    mine_routes = route_proba(mine_p, meta["intents"]).idxmax(axis=1)
    print(score("your fine-tuned encoder", validation, mine_routes, np.array(meta["intents"])[mine_p.argmax(axis=1)]))
else:
    print("Skipped. The other cells use the ready checkpoint, made by this same code.")

## 10. The hosted LLM, recorded

A hosted LLM (Azure chat-small) chose an intent for a sample of questions: up to 10 per intent. The few-shot prompt has the 77 intents, grouped by route, and the 8 most similar training questions. The next cell builds the prompt for one question, so you can read it. Its answers were recorded once: you replay them, with no key.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# The prompt of the recorded LLM answers. The few-shot prompt adds the 8 most similar training questions.
groups = "\n".join(f"- {r}: " + ", ".join(mapping.intent[mapping.route == r]) for r in ROUTES)
SYSTEM = {
    "zero-shot": ("You sort the questions that customers send to an online bank. Choose exactly one intent from "
                  f"this list (grouped by the team that answers it):\n{groups}\n"
                  "Answer in JSON with the intent and your confidence from 0 to 1."),
    "few-shot": ("You sort the questions that customers send to an online bank. Choose exactly one intent from "
                 f"this list (grouped by the team that answers it):\n{groups}\n"
                 "You also get labelled examples of similar questions. "
                 "Answer in JSON with the intent and your confidence from 0 to 1."),
}
picker = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
picker_x = picker.fit_transform(train.text)

def examples_for(text, k=8):
    """The k training questions with the closest TF-IDF vectors (letters), with their intents."""
    sims = (picker_x @ picker.transform([text]).T).toarray().ravel()
    return [(train.text[i], train.intent[i]) for i in sorted(sims.argsort()[::-1][:k])]

def user_prompt(text, mode):
    if mode == "zero-shot":
        return f"Question:\n{text}"
    shown = "\n".join(f"- {t!r} -> {i}" for t, i in examples_for(text))
    return f"Labelled examples:\n{shown}\n\nQuestion:\n{text}"

print(user_prompt(test_sample.text.iloc[0], "few-shot"))

The LLM was recorded for a sample only, so the next cell compares every method on that same sample.

> **Check.** On the validation sample, the LLM's route accuracy is about 0.963. It costs about a tenth of a cent per question, and each answer takes over a second.

In [ ]:
def recorded(mode):
    """The recorded answers of the hosted LLM (chat-small), one line per question."""
    rows = [json.loads(x) for x in open(f"recordings/llm-{mode}.jsonl", encoding="utf-8")]
    return pd.DataFrame(rows).set_index("message_id")

few = recorded("few-shot")
PRICE_IN, PRICE_OUT = 0.10, 0.50   # chat-small, dollars per 1 million tokens (the course's price table)
llm_run = (few.tokens_in.mean() * PRICE_IN + few.tokens_out.mean() * PRICE_OUT) / 1e6 * 1000
llm_ms = few.latency_s.median() * 1000
print(f"few-shot: {len(few)} answers, all valid: {few.valid.all()} | tokens in {few.tokens_in.mean():.0f}, "
      f"out {few.tokens_out.mean():.0f} | {llm_run:.3f} dollars per 1,000 | median {llm_ms:.0f} ms")
v = few.loc[validation_sample.message_id]
llm_val = v.intent.map(ROUTE_OF).values
enc_intent_val = np.array(meta["intents"])[enc_p_val.argmax(axis=1)]
idx = validation_sample.index
rows_sample = [score(m, validation_sample, P.loc[idx].idxmax(axis=1), I[idx], run_per_1000=r, ms=ms)
               for m, P, I, r, ms in [
                   ("TF-IDF + LR", tfidf_val, tfidf_intent_val, cpu_cost(tfidf_ms), tfidf_ms),
                   ("embeddings (frozen) + LR", frozen_val, frozen.predict(E["validation"]), cpu_cost(frozen_ms), frozen_ms),
                   ("fine-tuned encoder", enc_val, enc_intent_val, cpu_cost(enc_ms), enc_ms)]]
rows_sample.append(score("LLM few-shot (recorded)", validation_sample, llm_val, v.intent.values, llm_run, llm_ms))
table(rows_sample)

## 11. The review threshold

A question whose route confidence is below a threshold goes to a person. Each review costs 1.50 dollars, but it removes the risk of a 4- or 20-dollar mistake. The next cell tries every threshold from 0.30 to 0.99 on the validation questions.

> **Predict.** Is the cheapest threshold near 0.5, or near 0.9?

> **Check.** The cheapest threshold is 0.75.

In [ ]:
# The review queue: a question whose route confidence is below the threshold goes to a person (1.50 dollars).
def with_review(P, threshold):
    """The routes, with "review" where the highest route probability is below the threshold."""
    return np.where(P.max(axis=1) >= threshold, P.idxmax(axis=1), "review")

thresholds = np.round(np.arange(0.30, 1.00, 0.01), 2)
curve = pd.DataFrame([{"threshold": t, **score("fine-tuned encoder", validation, with_review(enc_val, t))}
                      for t in thresholds])
best = curve.loc[curve.total_per_1000.idxmin()]
THRESHOLD = float(best.threshold)
print(f"lowest cost on validation: threshold {THRESHOLD:.2f}, review {best.review_share:.1%}, "
      f"{best.total_per_1000:.1f} dollars per 1,000 (no review: {curve.total_per_1000.iloc[0]:.1f} at 0.30)")
shown = curve.threshold.isin([0.3, 0.5, 0.7, THRESHOLD, 0.9, 0.95, 0.99])
curve.loc[shown, ["threshold", "route_accuracy", "review_share", "mistakes_per_1000", "total_per_1000"]]

> **Your turn.** The review team can read at most 2% of the questions. Find the cheapest threshold under that limit, from `curve`.

In [ ]:
# The review team can read at most 2% of the questions. Find the cheapest threshold under that limit.
threshold_2 = ...   # a value from curve.threshold

In [ ]:
expect('threshold_2', threshold_2, 0.66)

## 12. The test, once

Now the test questions, for every method. Nothing changes after this cell.

> **Check.** The fine-tuned encoder reaches a route accuracy of about 0.976. With the review threshold 0.75, it costs about 92.4 dollars per 1,000 questions and sends 2.2% to a person.

In [ ]:
# The test questions, once. Every method was chosen on the validation questions; nothing changes after this cell.
tfidf_test = route_proba(tfidf.predict_proba(test.text), tfidf.classes_)
frozen_test = route_proba(frozen.predict_proba(E["test"]), frozen.classes_)
enc_p_test = encoder_proba(list(test.text))
enc_test = route_proba(enc_p_test, meta["intents"])
tfidf_intent_test, frozen_intent_test = tfidf.predict(test.text), frozen.predict(E["test"])
enc_intent_test = np.array(meta["intents"])[enc_p_test.argmax(axis=1)]
rows_test = [score("everything to a person", test, ["review"] * len(test)),
             score("TF-IDF + LR", test, tfidf_test.idxmax(axis=1), tfidf_intent_test, cpu_cost(tfidf_ms), tfidf_ms),
             score("embeddings (frozen) + LR", test, frozen_test.idxmax(axis=1), frozen_intent_test,
                   cpu_cost(frozen_ms), frozen_ms),
             score("fine-tuned encoder", test, enc_test.idxmax(axis=1), enc_intent_test, cpu_cost(enc_ms), enc_ms),
             score(f"fine-tuned encoder, review below {THRESHOLD:.2f}", test, with_review(enc_test, THRESHOLD),
                   run_per_1000=cpu_cost(enc_ms), ms=enc_ms)]
table(rows_test)

And on the test sample with the recorded LLM answers:

> **Check.** The few-shot LLM's route accuracy is about 0.983, and the zero-shot LLM's about 0.931.

In [ ]:
# The LLM was recorded for a sample of the test questions: compare every method on that same sample.
t = few.loc[test_sample.message_id]
zero = recorded("zero-shot").loc[test_sample.message_id]
zero_run = (zero.tokens_in.mean() * PRICE_IN + zero.tokens_out.mean() * PRICE_OUT) / 1e6 * 1000
idx = test_sample.index
rows_test_sample = [
    score("TF-IDF + LR", test_sample, tfidf_test.loc[idx].idxmax(axis=1), tfidf_intent_test[idx], cpu_cost(tfidf_ms)),
    score("embeddings (frozen) + LR", test_sample, frozen_test.loc[idx].idxmax(axis=1), frozen_intent_test[idx],
          cpu_cost(frozen_ms)),
    score("fine-tuned encoder", test_sample, enc_test.loc[idx].idxmax(axis=1), enc_intent_test[idx], cpu_cost(enc_ms)),
    score(f"fine-tuned encoder, review below {THRESHOLD:.2f}", test_sample, with_review(enc_test.loc[idx], THRESHOLD),
          run_per_1000=cpu_cost(enc_ms)),
    score("LLM zero-shot (recorded)", test_sample, zero.intent.map(ROUTE_OF).fillna("review").values,
          zero.intent.values, zero_run, zero.latency_s.median() * 1000),
    score("LLM few-shot (recorded)", test_sample, t.intent.map(ROUTE_OF).values, t.intent.values, llm_run, llm_ms)]
table(rows_test_sample)

### Would the LLM help with the unsure questions?

A cascade: the encoder decides when its route confidence is at least 0.75; below that, the LLM decides when its written confidence is high enough; a person reads the rest. The LLM threshold is chosen on the validation sample, then the cascade runs on the test sample.

> **Check.** Only 17 validation sample questions are unsure, so the chosen LLM threshold rests on very few questions. Read the lesson page before you trust the cascade's row.

In [ ]:
# Would the LLM help with the questions that the encoder is unsure about? A cascade: the encoder decides at or above
# THRESHOLD; below it, the LLM decides if its written confidence is at or above llm_sure; a person reads the rest.
def cascade(part, P, answers, llm_sure):
    sure = P.max(axis=1).to_numpy() >= THRESHOLD
    llm_route = answers.loc[part.message_id].intent.map(ROUTE_OF).to_numpy()
    llm_ok = answers.loc[part.message_id].confidence.to_numpy() >= llm_sure
    routes = np.where(sure, P.idxmax(axis=1), np.where(llm_ok, llm_route, "review"))
    calls = (~sure).mean() * 1000    # LLM calls per 1,000 questions
    return routes, calls

vs = validation_sample.index
search = []
for llm_sure in [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99, 1.01]:
    routes, calls = cascade(validation_sample, enc_val.loc[vs], few, llm_sure)
    search.append({"llm_sure": llm_sure, **score("cascade", validation_sample, routes, run_per_1000=calls * llm_run / 1000)})
search = pd.DataFrame(search)
LLM_SURE = float(search.loc[search.total_per_1000.idxmin(), "llm_sure"])
print(f"unsure validation sample questions: {(enc_val.loc[vs].max(axis=1) < THRESHOLD).sum()} of {len(vs)}; "
      f"chosen LLM threshold: {LLM_SURE}")
routes, calls = cascade(test_sample, enc_test.loc[test_sample.index], few, LLM_SURE)
rows_test_sample.append(score(f"cascade: encoder >= {THRESHOLD:.2f}, LLM >= {LLM_SURE}, else a person", test_sample,
                              routes, run_per_1000=calls * llm_run / 1000))
table(rows_test_sample)[["method", "route_accuracy", "review_share", "mistakes_per_1000", "run_per_1000", "total_per_1000"]]

## 13. Where the chosen model still fails

> **Check.** You should see 217 intent mistakes in 3,080 test questions, 142 of them inside one route.

In [ ]:
# Where does the chosen model still go wrong on the test questions?
pairs_test = pd.DataFrame({"intent": test.intent, "predicted": enc_intent_test})
pairs_test = pairs_test[pairs_test.intent != pairs_test.predicted].value_counts().rename("questions").reset_index()
pairs_test["same_route"] = pairs_test.intent.map(ROUTE_OF) == pairs_test.predicted.map(ROUTE_OF)
n_wrong = pairs_test.questions.sum()
print(f"intent mistakes: {n_wrong} of {len(test)}; inside the same route: {pairs_test.questions[pairs_test.same_route].sum()}")
route_pred = enc_test.idxmax(axis=1).values
routes_wrong = pd.crosstab(pd.Series(test.route.values, name="route"), pd.Series(route_pred, name="predicted"))
print("route mistakes:", int((route_pred != test.route.values).sum()))
pairs_test.head(8)

## 14. New questions

We wrote eight new questions, in the style of a real inbox. The router gives each one an intent, a route and a confidence, and sends the unsure ones to a person.

In [ ]:
# New input: questions that we wrote for this case study (not from Banking77).
new = pd.DataFrame({"text": [
    "My new card still has not come. It has been 10 days.",
    "There is a payment in Lisbon on my card and I have never been there.",
    "How long does a transfer to a euro account take?",
    "I paid cash in at the machine but my balance did not change.",
    "Why did I pay a fee when I topped up with my card?",
    "Can my 15 year old son open an account?",
    "The shop says the payment failed but the money left my account.",
    "hello?"]})
p = encoder_proba(list(new.text))
P = route_proba(p, meta["intents"])
new["intent"] = np.array(meta["intents"])[p.argmax(axis=1)]
new["route"], new["confidence"] = P.idxmax(axis=1), P.max(axis=1).round(3)
new["decision"] = np.where(new.confidence >= THRESHOLD, "send to " + new.route, "review by a person")
new

Read each row. Do you agree with the route? Which ones would you want a person to read?

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m10/route-banking-questions/) explains each step, the mapping, the costs, the recommendation and the limits, and how to make this case study a portfolio piece. Its knowledge checks count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Find what the bot cannot handle](https://learning.nextia-ai.com/courses/text-routing/m10/find-what-the-bot-cannot-handle/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m10/route-banking-questions/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Banking77 is by PolyAI, CC BY 4.0. Its questions were written for an imagined bank.